# Vicuña · Q2: supervisión auxiliar de las horas t-2 y t-1

Hecho a partir de `train/plantilla_experimento.ipynb` (PR #25), con el mismo cribado que Q1. Las celdas de `validacion.py` y `ventanas.py` son copias exactas.

**Hipótesis.** Con 2,935 muestras el límite es el sobreajuste. Si la red también predice el WRF de las horas t-2 y t-1 (etiquetas de train que ya tenemos), recibe más señal por muestra y aprende mejor la corrección del GFS, sin usar horas futuras.

**Un solo cambio** sobre r2: `aux = True`. Una cabeza aparte (convolución 1x1, 8 canales) sale de las mismas características del decodificador y predice t-2 y t-1, cada una con su residuo sobre el GFS de esa hora. Pérdida = pérdida de la hora t + 0.5 · pérdida auxiliar (misma fórmula 'rmse'). Solo se entrena con ventanas completas, así que t-2 y t-1 siempre existen y son horas de train. **En inferencia la cabeza auxiliar no se usa y no se guarda:** el `state_dict` tiene las mismas claves que r2 y sirve igual en `app/predict.py`.

**Base.** r2 con las ventanas completas (T11), entrenada en esta misma corrida con las mismas semillas. Entrada solo con t-2, t-1 y t del GFS.

**Regla fijada antes de correr.**
1. Cribado: corte A con semillas 0, 1 y 2. Pasa si la media de las diferencias pareadas con r2 es >= 0.002.
2. Si pasa, se completa a 5 semillas en A y en B (y r2 también).
3. Se acepta si el ensamble de 5 supera al de r2 en A y en B, y la media de las dos diferencias es >= 0.003.
4. Si se acepta, se reentrena con todo el train (5 semillas, ventanas completas) y deja pesos, `modelo.json` y CSV candidato. No se envía desde aquí.


In [ ]:
# ==== Tu experimento: edita solo esta celda ====
ID = 'Q2'                       # ID en el plan: E1, E2, ...
AUTOR = 'eduquispeperez'
HIPOTESIS = 'Predecir también el WRF de t-2 y t-1 (cabeza auxiliar) mejora la hora t'
CANDIDATOS = {                  # nombre -> UN solo cambio sobre BASE
    'aux0.5': dict(aux=True),
}
QUICK = False                   # True: prueba corta para ver que todo corre (los scores no valen)

# Base vigente: ronda 2 (residuo crudo, decodificador profundo, dropout 0.1)
BASE = {'w': 64, 'bloques': [1, 2, 4, 1], 'enc_prof': False, 'dropout': 0.1, 'wd': 0.0001, 'lags': 3, 'skip': False,
        'residuo': 'crudo', 'terreno': False, 'profundo': True, 'peso_mj': 1, 'epocas': 25,
        'lr': 2e-3, 'lote': 32, 'entradas': 'gfs', 'arq': 'base', 'ruido': 0.0, 'aux': False, 'peso_aux': 0.5}

In [ ]:
import glob, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F

def find(name):
    # Busca en Kaggle, en la carpeta data/ del repo (corriendo desde train/ o desde la raíz) y en la carpeta actual
    hits = (glob.glob(f'/kaggle/input/**/{name}', recursive=True) + glob.glob(f'../data/**/{name}', recursive=True)
            + glob.glob(f'data/**/{name}', recursive=True) + glob.glob(f'**/{name}', recursive=True))
    assert hits, f'No encuentro {name}'
    return hits[0]

tr = np.load(find('train.npz'))
te = np.load(find('test.npz'))
st = np.load(find('static.npz'))

Xtr, Ytr, ttr = tr['X'], tr['Y'], tr['times'].astype(str)
Xte, tte = te['X'], te['times'].astype(str)
dem = st['dem']
VARS = ['RH', 'T', 'U', 'V']; UNITS = ['%', 'K', 'm/s', 'm/s']

print('Train  X (GFS):', Xtr.shape, ' Y (WRF):', Ytr.shape, ' horas:', ttr[0], '→', ttr[-1])
print('Test   X (GFS):', Xte.shape, ' horas:', tte[0], '→', tte[-1])
print('DEM:', dem.shape, f'{dem.min():.0f}–{dem.max():.0f} m')

In [ ]:
# COPIA EXACTA de train/validacion.py: no editar aquí
"""
Validación temporal con la métrica del concurso (T02 · HU-09). Solo NumPy.

Corte temporal fijo: validación = últimas dos semanas de junio (desde 2024-06-16 00 UTC);
entrenamiento = todo lo anterior. Nunca se hace split aleatorio.

Hay dos scores y no son comparables entre sí:
  score_local    el del starter: cada RMSE se divide entre el RMSE de la climatología de train
                 medido en la propia validación (climatología = 0). Todas las horas.
  score_oficial  fórmula del leaderboard, con sus constantes fijas, sobre las horas que Kaggle
                 evalúa (00, 03, ..., 21 UTC). Es el que se usa para comparar experimentos.

El notebook lleva una copia exacta de este archivo en una celda para correr solo en Kaggle;
tests/test_validacion.py comprueba que ambas sean iguales.
"""
import numpy as np

VARS = ("RH", "T", "U", "V")
INICIO_VAL = "2024061600"
ESCALAS_OFICIALES = np.array([25.360, 2.0652, 2.6776, 2.0876])
HORAS_EVALUADAS = tuple(range(0, 24, 3))


def horas(times):
    """Hora UTC (0-23) de cada marca 'YYYYMMDDHH'."""
    return np.array([int(str(t)[-2:]) for t in times])


def corte_temporal(times, inicio_val=INICIO_VAL):
    """Máscaras booleanas (train, val): val son las marcas desde `inicio_val` en adelante."""
    val = np.asarray(times).astype(str) >= inicio_val
    trn = ~val
    if not trn.any() or not val.any():
        raise ValueError(f"El corte {inicio_val} deja vacío el entrenamiento o la validación.")
    return trn, val


def climatologia(Y, times):
    """Media por hora UTC: (24, filas, columnas, variables). Exige datos en las 24 horas."""
    h = horas(times)
    faltan = sorted(set(range(24)) - set(h.tolist()))
    if faltan:
        raise ValueError(f"No hay datos para las horas UTC {faltan}.")
    return np.stack([Y[h == k].mean(0) for k in range(24)])


def rmse_por_variable(pred, real):
    """RMSE de cada variable sobre todas las horas y celdas. Entradas (N, filas, columnas, 4)."""
    if pred.shape != real.shape:
        raise ValueError(f"Formas distintas: pred {pred.shape} y real {real.shape}.")
    rmse = np.sqrt(((pred - real) ** 2).mean(axis=(0, 1, 2)))
    if not np.isfinite(rmse).all():
        raise ValueError("Hay NaN o infinitos en la predicción o en el valor real.")
    return rmse


def score(rmse, escalas):
    """1 - media de rmse/escala; las cuatro variables pesan igual."""
    return float(1 - (rmse / escalas).mean())


def escala_climatologia(Y, times, inicio_val=INICIO_VAL):
    """RMSE en validación de la climatología calculada solo con train (denominador de score_local)."""
    t = np.asarray(times).astype(str)
    trn, val = corte_temporal(t, inicio_val)
    clim = climatologia(Y[trn], t[trn])
    return rmse_por_variable(clim[horas(t[val])], Y[val])


def validar(pred_val, Y, times, inicio_val=INICIO_VAL):
    """
    Evalúa una predicción de la validación.

    pred_val : (N_val, 70, 35, 4) en unidades físicas, en el orden de las horas de validación.
    Y, times : target y marcas de TODO train.npz (de ahí salen el corte y la climatología).
    """
    t = np.asarray(times).astype(str)
    _, val = corte_temporal(t, inicio_val)
    real, t_val = Y[val], t[val]
    rmse = rmse_por_variable(pred_val, real)
    evaluadas = np.isin(horas(t_val), HORAS_EVALUADAS)
    rmse_3h = rmse_por_variable(pred_val[evaluadas], real[evaluadas])
    return {
        "score_local": score(rmse, escala_climatologia(Y, t, inicio_val)),
        "rmse": {v: float(r) for v, r in zip(VARS, rmse)},
        "score_oficial": score(rmse_3h, ESCALAS_OFICIALES),
        "rmse_horas_evaluadas": {v: float(r) for v, r in zip(VARS, rmse_3h)},
        "horas_val": int(val.sum()),
        "horas_evaluadas": int(evaluadas.sum()),
        "primera": str(min(t_val)),
        "ultima": str(max(t_val)),
    }

In [ ]:
# COPIA EXACTA de train/ventanas.py: no editar aquí
"""
Ventanas (t-2, t-1, t) del GFS sin cruzar huecos de horas (T11 · HU-09). Solo NumPy.

La data del concurso tiene huecos de horas y de días. Una ventana está completa si existen las k horas
consecutivas que terminan en t. En las incompletas, la hora que falta se rellena con la hora t (igual que
el starter): sirven para predecir horas que Kaggle exige, pero se excluyen del entrenamiento.

Las horas previas se buscan por fecha, no por posición: nunca entra una hora del otro lado de un hueco.

El notebook lleva una copia exacta de este archivo en una celda para correr solo en Kaggle;
tests/test_ventanas.py comprueba que ambas sean iguales.
"""
from datetime import datetime, timedelta

import numpy as np


def indices_ventanas(times, k=3):
    """
    Índices de las horas (t-(k-1), …, t) de cada marca 'YYYYMMDDHH'.

    Devuelve (idx, completa): idx es (N, k), con el índice de t donde falta una hora previa;
    completa es una máscara (N,), True si las k horas existen.
    """
    fechas = [datetime.strptime(str(t), "%Y%m%d%H") for t in times]
    pos = {f: i for i, f in enumerate(fechas)}
    if len(pos) != len(fechas):
        raise ValueError("Hay horas repetidas en times.")
    idx = np.empty((len(fechas), k), np.int64)
    completa = np.ones(len(fechas), bool)
    for i, f in enumerate(fechas):
        for j, atras in enumerate(range(k - 1, -1, -1)):
            previa = pos.get(f - timedelta(hours=atras))
            if previa is None:
                completa[i] = False
                previa = i
            idx[i, j] = previa
    return idx, completa


def armar_ventanas(X, times, k=3):
    """Ventanas (N, k, ...) en float32 y la máscara de las completas."""
    if len(X) != len(times):
        raise ValueError(f"X tiene {len(X)} horas y times {len(times)}.")
    idx, completa = indices_ventanas(times, k)
    return np.asarray(X, np.float32)[idx], completa

In [ ]:
# La hora que falta en una ventana incompleta se rellena con la hora t; esas ventanas no entrenan (T11)
import functools

@functools.lru_cache(maxsize=None)
def ventanas_de(k):
    WTR, completa = armar_ventanas(Xtr, ttr, k)
    return WTR, completa

print('ventanas completas (3 h):', int(ventanas_de(3)[1].sum()), 'de', len(ttr))

In [ ]:
import json, platform
if not torch.cuda.is_available():
    print('ADVERTENCIA: sin GPU; solo sirve para QUICK')
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
EQUIPO = torch.cuda.get_device_name(0) if dev == 'cuda' else 'CPU'
print(EQUIPO, '·', platform.python_version(), np.__version__, torch.__version__)
# Dos cortes temporales: A (el de T02) y B (1-15 jun, sin mirar nada posterior)
VENTANAS = {'A': dict(usar=np.ones(len(ttr), bool), ini='2024061600'),
            'B': dict(usar=ttr < '2024061600', ini='2024060100')}
for k, v in VENTANAS.items():
    v['trn'] = v['usar'] & (ttr < v['ini']); v['val'] = v['usar'] & (ttr >= v['ini'])
    print(f"ventana {k}: train {v['trn'].sum()} h · val {v['val'].sum()} h · {ttr[v['val']][0]} → {ttr[v['val']][-1]}")
def to_t(a): return torch.tensor(np.asarray(a), dtype=torch.float32, device=dev)
ESC = to_t(ESCALAS_OFICIALES)

In [ ]:
# Grilla WRF dentro de la grilla GFS (como en el starter): GFS fila 0 = norte, WRF y DEM fila 0 = sur
glat, glon = st['gfs_lats'], st['gfs_lons']
wlat, wlon = st['wrf_lats'], st['wrf_lons']
LAT, LON = np.meshgrid(wlat, wlon, indexing='ij')
gy = (LAT - glat[0]) / (glat[-1] - glat[0]) * 2 - 1
gx = (LON - glon[0]) / (glon[-1] - glon[0]) * 2 - 1
GRID = to_t(np.stack([gx, gy], -1)[None])                      # (1,70,35,2)
DEM = to_t((dem - dem.mean()) / dem.std())[None, None]

def media_local(a, k):
    """Media en una ventana k x k con borde reflejado."""
    p = np.pad(a, k // 2, mode='reflect')
    return np.lib.stride_tricks.sliding_window_view(p, (k, k)).mean((-1, -2))

def relieve(dem):
    """Pendiente (norte, este, magnitud), orientación (sin, cos) y relieve relativo (~15 y ~45 km)."""
    z = dem.astype(np.float64)
    dn, de = np.gradient(z)                                    # fila crece hacia el norte, columna al este
    mag = np.hypot(dn, de)
    unit = np.where(mag > 0, mag, 1.0)
    capas = [dn, de, mag, de / unit, dn / unit, z - media_local(z, 5), z - media_local(z, 15)]
    return np.stack([(c - c.mean()) / c.std() for c in capas])

TERRENO = to_t(relieve(dem))[None]                             # (1,7,70,35)

def interpolar(a):
    """(B,C,17,13) GFS -> (B,C,70,35) WRF, bilineal."""
    return F.grid_sample(a, GRID.expand(len(a), -1, -1, -1), align_corners=True)

class Bloque(nn.Module):
    def __init__(self, w, d, p=0.0):
        super().__init__()
        self.p = p
        self.f = nn.Sequential(nn.Conv2d(w, w, 3, padding=d, dilation=d), nn.GELU(),
                               nn.Conv2d(w, w, 3, padding=d, dilation=d))
    def forward(self, x):
        return F.gelu(x + F.dropout2d(self.f(x), self.p, self.training))

class Net(nn.Module):
    """Red de la ronda 2. `nv` = número de variables de entrada por hora."""
    def __init__(self, skip=False, terreno=False, profundo=False, w=64,
                 bloques=(1, 2, 4, 1), enc_prof=False, dropout=0.0, lags=3, nv=4, aux=False):
        super().__init__()
        self.skip, self.terreno = skip, terreno
        self.enc = nn.Sequential(nn.Conv2d(nv * lags + 2, w, 3, padding=1), nn.GELU(), nn.Conv2d(w, w, 3, padding=1), nn.GELU())
        self.static = nn.Parameter(torch.zeros(1, 8, 70, 35))
        self.enc2 = nn.Sequential(Bloque(w, 1, dropout), Bloque(w, 1, dropout)) if enc_prof else nn.Identity()
        c = w + 9 + (nv * lags if skip else 0) + (7 if terreno else 0)
        if profundo:
            self.dec = nn.Sequential(nn.Conv2d(c, w, 3, padding=1), nn.GELU(),
                                     *[Bloque(w, d, dropout) for d in bloques], nn.Conv2d(w, 4, 1))
        else:
            self.dec = nn.Sequential(nn.Conv2d(c, w, 3, padding=1), nn.GELU(),
                                     nn.Conv2d(w, w, 3, padding=1), nn.GELU(), nn.Conv2d(w, 4, 1))
        # Q2: cabeza auxiliar para t-2 y t-1 (8 canales = 2 horas x 4 variables); se crea al final para no
        # cambiar la inicialización del resto con la misma semilla, y no se guarda en el state_dict final
        self.aux = nn.Conv2d(w, 8, 1) if aux else None
    def forward(self, x, h, con_aux=False):      # x: (B,lags,17,13,4), h: (B,) hora UTC
        B = x.shape[0]
        x = x.permute(0, 1, 4, 2, 3).reshape(B, -1, 17, 13)
        ang = 2 * np.pi * h / 24
        hh = torch.stack([torch.sin(ang), torch.cos(ang)], 1)[..., None, None].expand(B, 2, 17, 13)
        partes = [interpolar(self.enc2(self.enc(torch.cat([x, hh], 1)))), DEM.expand(B, -1, -1, -1),
                  self.static.expand(B, -1, -1, -1)]
        if self.skip:
            partes.append(interpolar(x))
        if self.terreno:
            partes.append(TERRENO.expand(B, -1, -1, -1))
        f = self.dec[:-1](torch.cat(partes, 1))
        out = self.dec[-1](f).permute(0, 2, 3, 1)  # (B,70,35,4) anomalía normalizada
        if con_aux:                              # solo al entrenar: (B,2,70,35,4) para t-2 y t-1
            return out, self.aux(f).permute(0, 2, 3, 1).reshape(B, 70, 35, 2, 4).permute(0, 3, 1, 2, 4)
        return out

In [ ]:
# ==== Código propio (solo si tu cambio es de entradas o de red) ====
def entradas_gfs(W):
    """W: (..., 4) GFS en unidades físicas (RH, T, U, V). Devuelve (..., nv). El residuo siempre usa las 4 originales."""
    return W

ENTRADAS = {'gfs': entradas_gfs}          # E1: añade aquí tu función y usa dict(entradas='nombre')
ARQUITECTURAS = {'base': Net}             # E3/E4: define tu clase (mismos argumentos que Net) y regístrala

In [ ]:
def datos_de(mascara, cfg):
    """Normalización y climatología SOLO con las horas de `mascara`."""
    t = ttr[mascara]
    E = np.asarray(ENTRADAS[cfg['entradas']](Xtr[mascara]), np.float32)
    mu_x = E.mean((0, 1, 2)); sd_x = E.std((0, 1, 2))
    assert (sd_x > 0).all(), 'Hay una variable de entrada constante'
    clim = climatologia(Ytr[mascara], t)
    anom_sd = (Ytr[mascara] - clim[horas(t)]).std((0, 1, 2))
    return {'cfg': cfg, 'CLIM': to_t(clim), 'ASD': to_t(anom_sd),
            'stats': dict(mu_x=mu_x, sd_x=sd_x, clim=clim, anom_sd=anom_sd)}

def preparar_x(D, W, t):
    """Entrada normalizada, hora y GFS interpolado como término base del residuo."""
    s = D['stats']
    x = to_t((np.asarray(ENTRADAS[D['cfg']['entradas']](W), np.float32) - s['mu_x']) / s['sd_x'])
    h = to_t(horas(t)).long()
    partes = []
    for i in range(0, len(W), 256):
        gfs = to_t(W[i:i + 256, -1])
        b = interpolar(gfs.permute(0, 3, 1, 2)).permute(0, 2, 3, 1) - D['CLIM'][h[i:i + 256]]
        partes.append(b / D['ASD'])
    return x, h, torch.cat(partes)

def red_de(cfg, nv):
    claves = ('skip', 'terreno', 'profundo', 'w', 'bloques', 'enc_prof', 'dropout', 'lags')
    return ARQUITECTURAS[cfg['arq']](nv=nv, aux=cfg['aux'], **{k: cfg[k] for k in claves}).to(dev)

def pesos_inferencia(mod):
    """state_dict sin la cabeza auxiliar: mismas claves que r2 (compatible con app/predict.py)."""
    return {k: v for k, v in mod.state_dict().items() if not k.startswith('aux.')}

def entrenar(D, tensores, epocas, semilla):
    x, h, b, y = tensores[:4]
    cfg = D['cfg']; lote = cfg['lote']
    torch.manual_seed(semilla); torch.cuda.manual_seed_all(semilla)
    rng = np.random.default_rng(semilla)
    model = red_de(cfg, x.shape[-1])
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['wd'])
    n = len(x); pasos = epocas * int(np.ceil(n / lote))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg['lr'], total_steps=pasos)
    for ep in range(epocas):
        model.train()
        orden = torch.tensor(rng.permutation(n), device=dev)
        for s in range(0, n, lote):
            i = orden[s:s + lote]
            xi = x[i]
            if cfg['ruido'] > 0:                       # Q1: ruido gaussiano en la entrada normalizada
                xi = xi + cfg['ruido'] * torch.randn_like(xi)
            if cfg['aux']:
                p, pa = model(xi, h[i].float(), con_aux=True)
                pred = p + b[i]
            else:
                pred = model(xi, h[i].float()) + b[i]
            mse = ((pred - y[i]) ** 2).mean(dim=(0, 1, 2)) * (D['ASD'] / ESC) ** 2
            loss = torch.sqrt(mse + 1e-8).mean()       # pérdida 'rmse' de T08
            if cfg['aux']:                             # Q2: misma pérdida para t-2 y t-1
                ba, ya = tensores[4], tensores[5]
                mse_a = ((pa + ba[i] - ya[i]) ** 2).mean(dim=(0, 2, 3)) * (D['ASD'] / ESC) ** 2   # (2, 4)
                loss = loss + cfg['peso_aux'] * torch.sqrt(mse_a + 1e-8).mean()
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return model

def tensores_de(D, mascara):
    W, completa = ventanas_de(D['cfg']['lags'])
    usar = np.where(mascara & completa)[0]
    if QUICK:
        usar = usar[:64]
    x, h, b = preparar_x(D, W[usar], ttr[usar])
    y = (to_t(Ytr[usar]) - D['CLIM'][h]) / D['ASD']
    # Q2: etiquetas y término base de t-2 y t-1. Con ventanas completas ambas horas existen y son de train
    assert D['cfg']['lags'] == 3
    idx, comp = indices_ventanas(ttr, 3)
    assert comp[usar].all()
    previas = idx[usar][:, :2].reshape(-1)                  # t-2, t-1 de cada muestra
    assert mascara[previas].all(), 'una hora previa cae fuera del entrenamiento'
    hp = to_t(horas(ttr[previas])).long()
    partes_y, partes_b = [], []
    for j in range(0, len(previas), 256):
        sl = slice(j, j + 256)
        partes_y.append((to_t(Ytr[previas[sl]]) - D['CLIM'][hp[sl]]) / D['ASD'])
        g = to_t(Xtr[previas[sl]])
        partes_b.append((interpolar(g.permute(0, 3, 1, 2)).permute(0, 2, 3, 1) - D['CLIM'][hp[sl]]) / D['ASD'])
    ya = torch.cat(partes_y).view(len(usar), 2, 70, 35, 4)
    ba = torch.cat(partes_b).view(len(usar), 2, 70, 35, 4)
    return x, h, b, y, ba, ya

def predecir_lote(model, D, W, t, bs=128):
    x, h, b = preparar_x(D, W, t); model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(W), bs):
            s = slice(i, i + bs)
            p = model(x[s], h[s].float()) + b[s]
            out.append((D['CLIM'][h[s]] + p * D['ASD']).cpu().numpy())
    P = np.concatenate(out)
    P[..., 0] = np.clip(P[..., 0], 0, 100)
    return P

def medir(P, k):
    v = VENTANAS[k]
    r = validar(P, Ytr[v['usar']], ttr[v['usar']], inicio_val=v['ini'])
    return {'score_oficial': r['score_oficial'], 'score_local': r['score_local'], 'rmse_3h': r['rmse_horas_evaluadas']}

def linea(etiqueta, m):
    v = ' '.join(f'{k} {x:.3f}' for k, x in m['rmse_3h'].items())
    print(f"{etiqueta:34s} oficial {m['score_oficial']:.5f} · local {m['score_local']:.5f} · {v}", flush=True)

In [ ]:
SEMILLAS_CRIBA = [0] if QUICK else [0, 1, 2]
SEMILLAS = [0, 1] if QUICK else [0, 1, 2, 3, 4]
UMBRAL_CRIBA, UMBRAL = 0.002, 0.003
SALIDA = f'exp_{ID}'
os.makedirs(SALIDA, exist_ok=True)
INICIO = time.time()

assert 'base' not in CANDIDATOS
for nombre, cambio in CANDIDATOS.items():
    assert len(cambio) == 1, f'{nombre}: un solo cambio por candidato'
    assert set(cambio) <= set(BASE), f'{nombre}: clave desconocida {set(cambio) - set(BASE)}'
    assert any(BASE[c] != x for c, x in cambio.items()), f'{nombre}: es igual a la base'

# Normalización, climatología y tensores por corte: no dependen de los candidatos de Q1
CACHE = {}
for k, v in VENTANAS.items():
    D = datos_de(v['trn'], BASE)
    CACHE[k] = (D, tensores_de(D, v['trn']), ventanas_de(BASE['lags'])[0][v['val']])

PRED, MED = {}, {}                      # (nombre, corte, semilla) -> predicción de validación / métricas
def correr(nombre, cfg, k, semilla):
    if (nombre, k, semilla) in MED:
        return
    D, T, W = CACHE[k]
    D = {**D, 'cfg': cfg}
    t0 = time.time()
    mod = entrenar(D, T, 1 if QUICK else cfg['epocas'], semilla)
    P = predecir_lote(mod, D, W, ttr[VENTANAS[k]['val']])
    m = medir(P, k)
    PRED[(nombre, k, semilla)], MED[(nombre, k, semilla)] = P.astype(np.float32), m
    torch.save(pesos_inferencia(mod), f'{SALIDA}/{nombre}_{k}_s{semilla}.pt')
    linea(f'{nombre} {k} s{semilla} ({time.time() - t0:.0f}s, total {time.time() - INICIO:.0f}s)', m)

CFG = {'base': dict(BASE), **{n: {**BASE, **c} for n, c in CANDIDATOS.items()}}

# 1) Cribado en el corte A, pareado por semilla
for s in SEMILLAS_CRIBA:
    correr('base', CFG['base'], 'A', s)
registro = {'base': {'cfg': CFG['base']}}
for nombre in CANDIDATOS:
    for s in SEMILLAS_CRIBA:
        correr(nombre, CFG[nombre], 'A', s)
    difs = [MED[(nombre, 'A', s)]['score_oficial'] - MED[('base', 'A', s)]['score_oficial'] for s in SEMILLAS_CRIBA]
    criba = float(np.mean(difs))
    registro[nombre] = {'cfg': CFG[nombre], 'criba_difs': difs, 'criba': criba, 'pasa_criba': bool(criba >= UMBRAL_CRIBA)}
    print(f'cribado {nombre}: diferencias pareadas {np.round(difs, 5)} · media {criba:+.5f} · pasa: {criba >= UMBRAL_CRIBA}', flush=True)

# 2) Completar a 5 semillas en A y B lo que pasó el cribado (en QUICK se completa uno para probar el código)
completar = [n for n in CANDIDATOS if registro[n]['pasa_criba']] or ([list(CANDIDATOS)[0]] if QUICK else [])
def ensamble(nombre, k):
    P = np.mean([PRED[(nombre, k, s)] for s in SEMILLAS], 0)
    np.savez(f'{SALIDA}/{nombre}_{k}_val.npz', times=ttr[VENTANAS[k]['val']], P=P.astype(np.float32))
    return medir(P, k)
if completar:
    for nombre in ['base'] + completar:
        for k in VENTANAS:
            for s in SEMILLAS:
                correr(nombre, CFG[nombre], k, s)
        registro[nombre]['ens'] = {k: ensamble(nombre, k) for k in VENTANAS}
        registro[nombre]['solas'] = {k: [MED[(nombre, k, s)]['score_oficial'] for s in SEMILLAS] for k in VENTANAS}
        for k in VENTANAS:
            linea(f'ensamble {nombre} {k}', registro[nombre]['ens'][k])
    for nombre in completar:
        r = registro[nombre]
        r['dif'] = {k: r['ens'][k]['score_oficial'] - registro['base']['ens'][k]['score_oficial'] for k in VENTANAS}
        r['dif_rmse'] = {k: {x: r['ens'][k]['rmse_3h'][x] - registro['base']['ens'][k]['rmse_3h'][x] for x in VARS} for k in VENTANAS}
        r['media'] = float(np.mean(list(r['dif'].values())))
        r['aceptado'] = bool(r['media'] >= UMBRAL and all(d > 0 for d in r['dif'].values()))
        print(f"{nombre}: ensamble - base {r['dif']} · media {r['media']:+.5f} · se acepta: {r['aceptado']}", flush=True)
for nombre in CANDIDATOS:
    registro[nombre].setdefault('aceptado', False)

ACEPTADOS = [n for n in CANDIDATOS if registro[n]['aceptado']]
json.dump({'id': ID, 'autor': AUTOR, 'hipotesis': HIPOTESIS, 'quick': QUICK, 'usa_horas_futuras': False,
           'semillas_criba': SEMILLAS_CRIBA, 'semillas': SEMILLAS, 'umbral_criba': UMBRAL_CRIBA, 'umbral': UMBRAL,
           'cribado_base_A': {s: MED[('base', 'A', s)] for s in SEMILLAS_CRIBA}, 'resultados': registro, 'aceptados': ACEPTADOS,
           'versiones': {'python': platform.python_version(), 'numpy': np.__version__, 'torch': torch.__version__, 'equipo': EQUIPO}},
          open(f'{SALIDA}/resultados_{ID.lower()}.json', 'w'), indent=1, default=float)
print('aceptados:', ACEPTADOS, f'· total {time.time() - INICIO:.0f}s', flush=True)


In [ ]:
# Reentrenamiento con TODOS los datos (ventanas completas, T11) de lo aceptado; en QUICK, del primer candidato
import hashlib
finales = ACEPTADOS or ([list(CANDIDATOS)[0]] if QUICK else [])
sample = pd.read_csv(find('sample_submission.csv'))
pos = {k: i for i, k in enumerate(tte)}
parts = sample['id'].str.split('_', expand=True)
ti = parts[0].map(pos).values; fr = parts[1].astype(int).values; co = parts[2].astype(int).values
todo = np.ones(len(ttr), bool)
for nombre in finales:
    cfg = CFG[nombre]
    carpeta = f'models/{ID.lower()}_{nombre}'
    os.makedirs(carpeta, exist_ok=True)
    D = datos_de(todo, cfg); T = tensores_de(D, todo)
    WTE_ = armar_ventanas(Xte, tte, cfg['lags'])[0]
    suma, miembros = 0, []
    for s in SEMILLAS:
        mod = entrenar(D, T, 1 if QUICK else cfg['epocas'], s)
        suma = suma + predecir_lote(mod, D, WTE_, tte)
        f = f'{carpeta}/{nombre}_s{s}.pt'
        torch.save(pesos_inferencia(mod), f)
        miembros.append({'archivo': os.path.basename(f), 'sha256': hashlib.sha256(open(f, 'rb').read()).hexdigest(), 'semilla': s})
    np.savez(f'{carpeta}/estadisticas.npz', **D['stats'])
    json.dump({'variante': 'residuo_crudo_profundo', 'experimento': ID, 'candidato': nombre, 'config': cfg,
               'ventanas': 'completas (T11)', 'usa_horas_futuras': False, 'cabeza_auxiliar_guardada': False, 'miembros': miembros,
               'normalizacion': {'mu_x': D['stats']['mu_x'].tolist(), 'sd_x': D['stats']['sd_x'].tolist()}},
              open(f'{carpeta}/modelo.json', 'w'), indent=1)
    P = suma / len(SEMILLAS)
    sub = pd.DataFrame({'id': sample['id']})
    for j, v in enumerate(VARS):
        sub[v] = P[ti, fr, co, j].round(3)
    assert len(sub) == len(sample) == 646800 and np.isfinite(sub[list(VARS)].values).all()
    sub.to_csv(f'submission_{ID.lower()}_{nombre}.csv', index=False)
    print(f'submission_{ID.lower()}_{nombre}.csv', sub.shape, f'· total {time.time() - INICIO:.0f}s', flush=True)
if not finales:
    print('Ningún candidato se acepta: no se reentrena con todos los datos')
